# 01. Genomic variation and association profiles

Describe variant composition and chromosome profiles, then examine association signals relative to a regional reference.

# 01. 基因组变异与关联信号分布

描述变异组成及染色体上的分布，再考察关联信号相对于区域参照的变化。

## 1. Variant data


## 1. 变异数据


### 1.1 Analysis setup


### 1.1 分析准备


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import socket
import sys
from datetime import datetime
from time import perf_counter
import pandas as pd

%load_ext autoreload
%autoreload 2

project_dir = '/absolute/remote/project/root'  # Replace with the confirmed config path.
os.chdir(project_dir)
project_dir = os.getcwd()
data_dir = f'{project_dir}/data'
source_dir = f'{data_dir}/source'
results_dir = f'{data_dir}/results'

os.makedirs(data_dir, exist_ok=True)
os.makedirs(source_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)

from src.plot_style import set_plot_style
set_plot_style()

print(f'Host: {socket.gethostname()}')
print(f'Working directory: {os.getcwd()}')
print(f'Python: {sys.executable}')
print(f"Environment prefix: {os.environ.get('CONDA_PREFIX', '<not set>')}")

### 1.2 Variant coordinates and allele frequencies

Use variant coordinates and minor allele frequencies (MAF) to describe genetic variation.


### 1.2 变异坐标与等位基因频率

利用变异坐标和次要等位基因频率（MAF）描述遗传变异。


In [ ]:
variant_file = f'{source_dir}/variants.csv'
variant_table_h5 = f'{results_dir}/variant_table.h5'
start_time = perf_counter()

if os.path.exists(variant_table_h5):
    variants_df = pd.read_hdf(variant_table_h5, key='df')
    print(f'[{datetime.now():%H:%M:%S}] Reusing: {variant_table_h5}')
else:
    variants_df = pd.read_csv(variant_file, dtype={'CHROM': str}, usecols=['CHROM', 'POS', 'MAF'])
    temporary_file = f'{variant_table_h5}.tmp'
    variants_df.to_hdf(temporary_file, key='df', mode='w')
    os.replace(temporary_file, variant_table_h5)
    print(f'[{datetime.now():%H:%M:%S}] Saved: {variant_table_h5}')

print(f'Variants: {len(variants_df):,}; elapsed: {perf_counter() - start_time:.1f}s')
variants_df.head()

## 2. Window-level variant summaries

Summarize variant counts and mean MAF in genomic windows to describe their chromosomal distribution.


## 2. 窗口水平的变异汇总

在基因组窗口中汇总变异数量和平均 MAF，描述它们沿染色体的分布。


### 2.1 Window statistics

Retain variants with MAF ≥0.05 and summarize occupied, non-overlapping 500 bp windows using 1-based coordinates. Mean MAF is calculated over retained variants, not all bases; windows without retained variants are omitted.

### 2.1 窗口统计

保留 MAF ≥0.05 的变异，使用从 1 开始的坐标，在互不重叠的 500 bp 窗口中汇总有变异的窗口。平均 MAF 以保留变异为分母，而非所有碱基；不含保留变异的窗口不展示。

In [ ]:
window_size = 500
min_maf = 0.05
summary_file = f'{results_dir}/variant_windows.b{window_size}.maf{min_maf}.csv.gz'

- Summarize variants by chromosome and window


- 按染色体及窗口汇总变异


In [ ]:
from src.style_examples import build_chromosome_summaries

if os.path.exists(summary_file):
    summary_df = pd.read_csv(summary_file, dtype={'CHROM': str})
    print(f'[{datetime.now():%H:%M:%S}] Reusing: {summary_file}')
else:
    summary_df = build_chromosome_summaries(
        variants_df, window_size=window_size, min_maf=min_maf,
    )
    temporary_file = f'{summary_file}.tmp'
    summary_df.to_csv(temporary_file, index=False, compression='gzip')
    os.replace(temporary_file, summary_file)
    print(f'[{datetime.now():%H:%M:%S}] Saved: {summary_file}')

print(f'Occupied windows: {len(summary_df):,}; columns: {summary_df.columns.tolist()}')
summary_df.head()

## 3. Chromosome profiles

Compare the mean MAF of retained variants across windows on selected chromosomes.


## 3. 染色体上的分布

比较选定染色体上不同窗口中保留变异的平均 MAF。


In [ ]:
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from src.style_examples import plot_window_maf

if not os.path.isfile(summary_file):
    raise FileNotFoundError(f'Run section 2.1 calculation first: {summary_file}')
summary_df = pd.read_csv(summary_file, dtype={'CHROM': str})
chromosomes = ['Chr1', 'Chr2']  # Match identifiers in the input table.
figure_prefix = f'{results_dir}/variant_windows.b{window_size}.maf{min_maf}.chromosome_profiles'
fig, axes = plt.subplots(1, len(chromosomes), figsize=(12, 3), squeeze=False)
plot_tables = []

for chromosome, ax in tqdm(list(zip(chromosomes, axes[0])), desc='Plotting profiles', unit='chromosome'):
    plot_window_maf(summary_df, chromosome, ax)
    ax.set_ylim(0, 0.5)
    ax.set_title(chromosome)
    plot_df = summary_df.loc[summary_df['CHROM'] == chromosome].sort_values('WINDOW_START').copy()
    plot_df['POSITION_MB'] = plot_df['WINDOW_START'] / 1e6
    plot_df['PANEL_CHROM'] = chromosome
    plot_tables.append(plot_df)

fig.tight_layout()
pd.concat(plot_tables, ignore_index=True).to_csv(f'{figure_prefix}.csv.gz', index=False, compression='gzip')
fig.savefig(f'{figure_prefix}.pdf')
fig.savefig(f'{figure_prefix}.png', dpi=200)
plt.show()
print(f'Figure and data: {figure_prefix}.{{pdf,png,csv.gz}}')

**Figure 1. Window-level allele-frequency profiles on selected chromosomes.** Each panel shows mean MAF among variants with MAF ≥0.05 in occupied 500 bp windows. The horizontal axis gives window starts in Mb. Lines connect occupied windows across gaps.


**图 1. 选定染色体上窗口水平的等位基因频率分布。** 各面板展示有变异的 500 bp 窗口中 MAF ≥0.05 变异的平均 MAF。横坐标表示窗口起点，单位为 Mb。折线跨越空缺连接有变异的窗口。


## 4. Variant composition

Compare the numbers of retained variants across chromosomes.

## 4. 变异组成

比较不同染色体上保留变异的数量。

In [ ]:
composition_csv = f'{results_dir}/variant_counts.maf{min_maf}.csv.gz'
composition_figure = f'{results_dir}/variant_counts.maf{min_maf}'

- Count retained variants

- 统计保留变异数量

In [ ]:
variants = pd.read_hdf(variant_table_h5, key='df')
counts = variants.loc[variants['MAF'] >= min_maf].groupby('CHROM').size()
counts_df = counts.rename('N_VARIANTS').reset_index()
counts_df.to_csv(composition_csv, index=False, compression='gzip')
counts_df

In [ ]:
import matplotlib.pyplot as plt

plot_df = pd.read_csv(composition_csv, dtype={'CHROM': str})
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(plot_df['CHROM'], plot_df['N_VARIANTS'], color='#457b9d')
ax.set(xlabel='Chromosome', ylabel='Retained variants')
fig.tight_layout()
plot_df.to_csv(f'{composition_figure}.plot_data.csv.gz', index=False, compression='gzip')
fig.savefig(f'{composition_figure}.pdf')
plt.show()

**Figure 2. Numbers of variants with MAF ≥0.05 on each chromosome.**

**图 2. 各染色体上 MAF ≥0.05 的变异数量。**

## 5. Regional association profiles

Compare local association signals with the surrounding region while keeping the original variant-level evidence visible.

## 5. 区域关联信号分布

将局部关联信号与周围区域比较，同时展示原始位点水平的关联证据。

### 5.1 Relative association signal intensity

For one predefined anchor, summarize raw association P values in 500 bp windows at 10 bp steps across 10 kb flanks. Relative association signal intensity (RASI) is

$$\mathrm{RASI}_w = \frac{-\log_{10} Q_{0.25}(P_i\mid i\in w)}{-\log_{10} Q_{0.25}(P_i\mid i\in R)},$$

where $R$ is the full anchor region and $w$ is a complete window inside it. Quantiles are calculated on raw P values before transformation. Retain finite values with $0<P\leq1$. If the region has fewer than four valid variants, or its transformed quartile is zero, RASI is undefined. Otherwise, windows with fewer than four valid variants use the regional quartile and have RASI=1.

### 5.1 相对关联信号强度

围绕一个预先定义的锚点，在两侧各 10 kb 内以 10 bp 步长划分 500 bp 窗口，汇总原始关联 P 值。相对关联信号强度（RASI）定义为

$$\mathrm{RASI}_w = \frac{-\log_{10} Q_{0.25}(P_i\mid i\in w)}{-\log_{10} Q_{0.25}(P_i\mid i\in R)},$$

其中 $R$ 是完整锚点区域，$w$ 是完全位于该区域内的窗口。先对原始 P 值取分位数，再进行变换。仅保留有限且满足 $0<P\leq1$ 的值。若区域内有效变异少于四个，或区域四分位数变换后为零，则 RASI 未定义。否则，有效变异少于四个的窗口使用区域四分位数，RASI=1。

RASI >1 indicates stronger local association signals than the regional reference; RASI <1 indicates weaker signals. This descriptive ratio is not a significance test and is not an effect-size estimate.

RASI >1 表示局部关联信号强于区域参照，RASI <1 表示较弱。该描述性比值不是显著性检验，也不是效应量估计。

In [ ]:
association_csv = f'{source_dir}/associations.csv.gz'
anchors_csv = f'{source_dir}/gene_anchors.csv'
# Associations: CHROM, POS (1-based), P_VALUE; one row per variant.
# Anchors: ANCHOR_ID, CHROM, POS (1-based), STRAND (+ or -).
anchor_id = 'gene_A_TSS'  # Select an actual input identifier.
flank = 10000
bin_size = 500
step = 10
min_variants = 4
profile_prefix = f'{results_dir}/{anchor_id}.f{flank}.b{bin_size}.s{step}.n{min_variants}.rasi'
sites_csv = f'{profile_prefix}.sites.csv.gz'
windows_csv = f'{profile_prefix}.windows.csv.gz'
region_csv = f'{profile_prefix}.region.csv.gz'
force_recompute = False

- Summarize regional and window association signals

- 汇总区域和窗口关联信号

In [ ]:
import numpy as np
from tqdm.auto import tqdm

# Reuse assumes these input files and their scientific definitions are unchanged.
expected_outputs = [sites_csv, windows_csv, region_csv]
stage_start = perf_counter()
if not force_recompute and all(os.path.isfile(path) for path in expected_outputs):
    print(f'[{datetime.now():%H:%M:%S}] Reusing: {profile_prefix}')
else:
    anchors = pd.read_csv(anchors_csv, dtype={'CHROM': str}).set_index('ANCHOR_ID')
    if not anchors.index.is_unique:
        raise ValueError('Anchor IDs must be unique.')
    anchor = anchors.loc[anchor_id]
    if anchor['STRAND'] not in ['+', '-']:
        raise ValueError('STRAND must be + or -.')
    direction = 1 if anchor['STRAND'] == '+' else -1
    sites = pd.read_csv(association_csv, dtype={'CHROM': str})
    sites = sites.loc[sites['CHROM'] == anchor['CHROM']].copy()
    sites['RELATIVE_POS'] = (sites['POS'] - anchor['POS']) * direction
    sites = sites.loc[sites['RELATIVE_POS'].between(-flank, flank)].copy()
    valid = np.isfinite(sites['P_VALUE']) & sites['P_VALUE'].between(0, 1, inclusive='right')
    sites = sites.loc[valid].sort_values('RELATIVE_POS').copy()
    sites['SIGNAL'] = -np.log10(sites['P_VALUE'])
    n_region = len(sites)
    region_q25 = sites['P_VALUE'].quantile(0.25) if n_region >= min_variants else np.nan
    region_signal = -np.log10(region_q25)
    defined_reference = np.isfinite(region_signal) and region_signal > 0

    starts = range(-flank, flank - bin_size + 2, step)
    rows = []
    for start in tqdm(starts, desc='Association windows', unit='window'):
        end = start + bin_size - 1
        window_p = sites.loc[sites['RELATIVE_POS'].between(start, end), 'P_VALUE']
        n_window = len(window_p)
        fallback = n_window < min_variants
        window_q25 = region_q25 if fallback else window_p.quantile(0.25)
        window_signal = -np.log10(window_q25)
        rasi = window_signal / region_signal if defined_reference else np.nan
        rows.append((start, end, (start + end) / 2, n_window, fallback, window_signal, rasi))
    windows = pd.DataFrame(rows, columns=[
        'WINDOW_START', 'WINDOW_END', 'RELATIVE_CENTER', 'N_VARIANTS',
        'USES_REGION_REFERENCE', 'WINDOW_SIGNAL', 'RASI',
    ])
    region = pd.DataFrame([dict(
        ANCHOR_ID=anchor_id, N_VARIANTS=n_region, REGION_P_Q25=region_q25,
        REGION_SIGNAL=region_signal, REFERENCE_DEFINED=defined_reference,
    )])
    for table, path in [(sites, sites_csv), (windows, windows_csv), (region, region_csv)]:
        temporary_path = f'{path}.tmp'
        table.to_csv(temporary_path, index=False, compression='gzip')
        os.replace(temporary_path, path)
    print(f'[{datetime.now():%H:%M:%S}] Saved: {profile_prefix}')
print(f'Elapsed: {perf_counter() - stage_start:.1f}s')
pd.read_csv(region_csv)

### 5.2 Variant evidence and relative profiles

### 5.2 位点证据与相对信号分布

In [ ]:
import matplotlib.pyplot as plt

for path in [sites_csv, windows_csv, region_csv]:
    if not os.path.isfile(path):
        raise FileNotFoundError(f'Run section 5.1 calculation first: {path}')
site_plot = pd.read_csv(sites_csv)
window_plot = pd.read_csv(windows_csv)
region_plot = pd.read_csv(region_csv)
reference = region_plot.loc[0, 'REGION_SIGNAL']
fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
axes[0].scatter(site_plot['RELATIVE_POS'], site_plot['SIGNAL'], s=6, color='#457b9d')
axes[0].set_ylabel('−log10(P)')
axes[1].plot(window_plot['RELATIVE_CENTER'], window_plot['WINDOW_SIGNAL'], color='#457b9d')
axes[1].set_ylabel('Quartile signal')
axes[2].plot(window_plot['RELATIVE_CENTER'], window_plot['RASI'], color='#457b9d')
axes[2].set(ylabel='RASI', xlabel='Distance from anchor (bp)')
if pd.notna(reference):
    axes[1].axhline(reference, color='#e09f3e', linestyle='--')
axes[2].axhline(1, color='#e09f3e', linestyle='--')
for ax in axes:
    ax.axvline(0, color='gray', linestyle=':')
    ax.set_xlim(-flank, flank)
fig.suptitle(anchor_id)
fig.tight_layout()
site_plot.to_csv(f'{profile_prefix}.figure_sites.csv.gz', index=False, compression='gzip')
window_plot.to_csv(f'{profile_prefix}.figure_windows.csv.gz', index=False, compression='gzip')
region_plot.to_csv(f'{profile_prefix}.figure_reference.csv.gz', index=False, compression='gzip')
fig.savefig(f'{profile_prefix}.pdf')
plt.show()

**Figure 3. Variant-level association evidence and relative signals around the selected anchor.** From top to bottom, panels show variant −log10(P), the transformed window P-value quartile, and RASI as defined in section 5.1. Blue points and curves show the observed values; orange dashed lines mark the regional quartile signal and RASI=1. Windows span 500 bp at 10 bp steps, and their centers are plotted. Coordinates follow transcription direction, with negative positions upstream; vertical dotted lines mark the anchor. Low-count windows use the regional reference. Where the regional reference is undefined, RASI is not shown.

**图 3. 选定锚点周围的位点关联证据与相对信号。** 从上到下，各面板展示位点 −log10(P)、窗口 P 值四分位数变换后的信号，以及第 5.1 节定义的 RASI。蓝色散点和曲线表示观测值；橙色虚线分别表示区域四分位数信号和 RASI=1。窗口宽 500 bp、步长 10 bp，按窗口中心位置绘制。坐标沿转录方向排列，负值表示上游；竖直点线标记锚点。低位点数窗口使用区域参照。区域参照未定义时不展示 RASI。